<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Operating-Systems/01-os-abstractions-and-kernel-boundary.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Operating Systems guideline](Operating-Systems.html)


## **Operating System Abstractions and the Kernel Boundary**

An ordinary program appears to own a processor, a continuous address space, files, and devices. The machine underneath provides none of those objects in that convenient form. It provides CPU execution modes, physical memory, storage blocks, interrupt lines, timers, and device registers. The operating system turns those mechanisms into stable abstractions and decides how many mutually untrusted programs may share them.

This chapter builds the boundary on which the rest of operating systems depends. The recurring example is:

```text
cat input.txt | grep kernel > result.txt
```

The command looks like one line of shell syntax. In reality it causes process creation, executable loading, file-descriptor manipulation, protected system calls, scheduling, memory translation, file-system access, and device I/O. This chapter first asks a narrower question: **what must happen whenever that command needs the kernel?**

### **What Does an Operating System Do?**

An **operating system** is the software environment that makes a computer usable as a shared, protected, and persistent system. The **kernel** is its privileged core. Shells, system libraries, service managers, and command-line utilities are usually part of the operating-system environment, but they normally execute in user mode and are not themselves the kernel.

This distinction matters. Calling `printf()` does not automatically mean entering the kernel: a C library may format text entirely in user space and buffer it. Eventually, however, emitting those bytes to a terminal or file requires a service such as `write()`, and that request crosses into the kernel.

![Basic placement of an operating system between applications and hardware](assets/os-placement.svg){fig-alt="A hierarchy showing users and applications above the operating system, with hardware below it." width="38%"}

*Figure source: Golftheman, [Operating system placement](https://commons.wikimedia.org/wiki/File:Operating_system_placement.svg), CC BY-SA 3.0.*

The operating system has several simultaneous responsibilities:

- **Convenience:** expose files instead of raw disk sectors, processes instead of an undivided CPU, and sockets instead of direct network-controller commands.
- **Multiplexing:** share finite processors, memory frames, storage bandwidth, and device time among competing activities.
- **Isolation:** prevent one program from reading, corrupting, or monopolizing another program's resources without authorization.
- **Coordination:** provide controlled ways for programs to synchronize, exchange data, and share persistent objects.
- **Recovery and accounting:** detect failures, reclaim resources, record ownership and usage, and restore consistent state when possible.

The kernel is therefore not merely a library of useful routines. A normal library trusts its caller and runs with the caller's authority. A kernel must assume that every user request may be malformed, stale, accidental, or hostile, while still serving valid requests efficiently.

#### **Abstraction, Multiplexing, and Isolation**

These three ideas describe the same resource from different viewpoints.

An **abstraction** gives software a simpler and more stable object than the hardware exposes. A file is a named byte sequence even though its contents may be scattered across storage blocks. A process is an execution environment containing registers, an address space, credentials, and open files even though its instructions occupy a physical CPU only intermittently.

**Multiplexing** maps many logical objects onto fewer physical resources. Time multiplexing lets many runnable threads take turns on a CPU. Space multiplexing assigns different memory frames or disk blocks to different owners. The mapping can change over time without changing the abstraction visible to a program.

**Isolation** constrains those mappings. Page-table permissions stop one process from casually addressing another process's memory. File permissions and credentials constrain which persistent objects a process may open. Preemptive scheduling prevents a user program from keeping the processor forever simply by refusing to return control voluntarily.

| Physical resource | OS abstraction | Multiplexing decision | Isolation rule |
|---|---|---|---|
| CPU cores | Processes and threads | Which runnable thread executes next, and for how long | User code can be preempted; privileged state is protected |
| Physical memory | Virtual address spaces and mappings | Which virtual pages occupy which frames | Page permissions and per-process mappings restrict access |
| Storage blocks | Files, directories, and volumes | Placement, caching, and I/O ordering | Ownership, permissions, namespaces, and access checks |
| Network interface | Sockets | Queueing packets among flows and processes | Binding rules, credentials, firewall policy, namespaces |
| Device registers | Device files and driver operations | Serialize or schedule device requests | Direct access is restricted to trusted code or delegated capabilities |

A useful mental model is **request -> check -> bind -> use -> release**. A process requests an object or operation; the kernel checks arguments and authority; it binds the request to kernel state; the process uses the resulting handle; and the kernel eventually releases or reclaims the state. File descriptors, virtual-memory mappings, sockets, and process identifiers all follow variations of this lifecycle.

The abstractions are also fault-containment boundaries. If `grep` dereferences an invalid pointer, the operating system can terminate `grep` while leaving the shell and unrelated processes alive. Without protected boundaries, an ordinary application bug could overwrite the scheduler, device driver, or another program.

#### **Mechanism and Policy**

A **mechanism** describes what the system is capable of doing. A **policy** chooses what it should do in a particular situation. Separating them lets the same low-level machinery support different goals.

| Area | Mechanism | Example policy question |
|---|---|---|
| CPU | Timer interrupt, context switch, run queues | Which thread should run next? |
| Memory | Page tables, faults, frame allocation | Which page should be evicted under pressure? |
| Storage | Block I/O, cache, journaling primitives | When should dirty data be written back? |
| Protection | User/kernel mode, page permissions, credentials | Which principal may perform this operation? |
| Networking | Packet queues and scheduling hooks | Which traffic receives priority or a rate limit? |

For example, a timer interrupt is a mechanism that lets the kernel regain control. Round-robin, priority scheduling, and fair scheduling are policies built on that mechanism. Page tables provide a mechanism for mapping and protecting memory; demand paging and page replacement are policies that decide when mappings are created and which frames are reclaimed.

The separation is a design principle, not an absolute law. A mechanism exposes some choices and hides others, so it always shapes policy. A scheduler that only supports one global queue cannot efficiently express every locality-aware policy. Good operating-system design therefore asks two questions together: *what decisions should remain configurable, and what mechanism makes those decisions enforceable?*

### **Hardware Foundations for Protection**

Software cannot create a trustworthy boundary using ordinary instructions alone. If every program could rewrite page tables, disable interrupts, or program storage devices directly, a malicious program could bypass any rule expressed by the kernel. Modern processors provide privilege modes, protected control registers, address translation, and controlled entry instructions so that the operating system can enforce its abstractions.

#### **User Mode and Kernel Mode**

The processor records a current **privilege level**. In user mode, code can perform ordinary computation but cannot execute selected privileged operations or access kernel-only mappings. In kernel mode, trusted code can configure the machine and access protected resources.

The familiar two-mode explanation is a simplification of architecture-specific designs:

- x86 defines four rings, numbered 0 through 3. Mainstream operating systems normally use ring 0 for the kernel and ring 3 for applications; rings 1 and 2 are rarely used by general-purpose kernels.
- RISC-V defines machine, supervisor, and user modes. Firmware commonly occupies machine mode, the operating-system kernel runs in supervisor mode, and applications run in user mode.
- Virtualization adds another control layer. A hypervisor can constrain a guest kernel even though that guest believes it is managing a machine.

![Privilege rings on x86](assets/x86-privilege-rings.svg){fig-alt="Concentric x86 privilege rings, with applications in the least privileged outer ring and the operating-system kernel in the most privileged inner ring." width="58%"}

*Figure source: Hertzsprung, [Privilege rings for x86](https://commons.wikimedia.org/wiki/File:Priv_rings.svg), CC BY-SA 3.0.*

Privilege is attached to the currently executing CPU context, not permanently to a source file or programming language. A C program normally runs in user mode. A kernel written in C normally runs in supervisor or kernel mode. The decisive difference is how execution was entered, which mappings and stacks are active, and what the processor permits at that moment.

Three terms are often confused:

| Transition | What changes | Must another process run? |
|---|---|---|
| Function call | Program counter, call stack, selected registers | No |
| Mode switch | Processor privilege and protected execution context | No |
| Context switch | Saved execution state changes from one thread/process to another | Yes |

A system call causes a mode switch, but it need not cause a context switch. If the requested operation completes immediately, the kernel may return to the same thread. If a `read()` must wait for data, the kernel can block that thread and schedule another one, adding a context switch.

#### **Privileged Instructions and Memory Protection**

Privileged instructions and registers control system-wide state. Exact details vary by architecture, but kernels typically need exclusive control over operations such as:

- installing or modifying page-table roots and memory-protection settings;
- configuring interrupt delivery, timers, and interrupt masks;
- entering low-power or halted states;
- accessing protected device-control interfaces;
- changing privilege-control and virtualization state.

Memory protection is especially important because kernel code and data must remain inaccessible while user code is running. The memory-management unit translates a virtual address through page tables and checks permissions such as readable, writable, executable, user-accessible, and present. A user load from a supervisor-only page does not simply return arbitrary bytes: the processor raises a fault and transfers control to the kernel.

This produces a layered defense:

1. The application can name only virtual addresses in its own address space.
2. Page-table entries determine whether those addresses are mapped and which accesses are legal.
3. User mode prevents the application from replacing the page tables with permissive ones.
4. The kernel decides whether a fault can be resolved, should become a user-visible signal, or indicates a kernel bug.

The timer completes the protection story. Even a program containing an infinite loop cannot indefinitely prevent the operating system from running, because a hardware timer can interrupt it and transfer control to the kernel. The scheduler may then return to the same thread or choose another.

The detailed hardware side of privilege, address translation, and interrupts is developed in [Processor Datapath and Control](../Computer-Organization/06-processor-datapath-and-control.html), [Virtual Memory and Address Translation](../Computer-Organization/09-virtual-memory-and-translation.html), and [I/O, Interrupts, and DMA](../Computer-Organization/10-io-interrupts-dma-storage.html). Here the focus is how an operating system uses those mechanisms.

### **Events That Enter the Kernel**

An operating system needs controlled ways to regain control of a CPU. The broad term **trap** is often used for any event that redirects execution to a privileged handler. Three major sources are system calls, synchronous exceptions, and asynchronous hardware interrupts.

![System calls, exceptions, and interrupts entering and returning from the kernel](assets/kernel-entry-and-return.svg){fig-alt="Three event types converge on architecture-defined trap entry, register saving, kernel handlers, state restoration, and protected return." width="96%"}

*Figure: original diagram for this chapter, based on the trap model in the [MIT xv6 book, Chapter 4](https://pdos.csail.mit.edu/6.1810/2025/xv6/book-riscv-rev5.pdf).*

| Property | System call | Exception or fault | Hardware interrupt |
|---|---|---|---|
| Immediate source | Current instruction deliberately requests a service | Current instruction violates a rule or needs kernel assistance | Timer, device, or another processor |
| Timing | Synchronous | Synchronous | Asynchronous to the current instruction stream |
| Expected by the program | Yes | Sometimes; often not | Usually no |
| Typical examples | `read`, `write`, `mmap`, `fork` | Page fault, illegal instruction, divide error | Timer tick, keyboard input, completed disk or network I/O |
| Possible outcome | Return value or error | Retry, signal/termination, or kernel failure | Resume, wake a waiter, or trigger scheduling |

#### **System Calls**

A **system call** is an intentional request for a kernel service. User code places a system-call number and arguments where the platform ABI requires, then executes a special instruction such as x86-64 `syscall` or RISC-V `ecall`. Hardware raises privilege and transfers control to a configured entry point; it does not let the application choose an arbitrary kernel function.

The kernel uses the system-call number to dispatch to a handler. The handler validates the request, performs or initiates the operation, stores a result, and eventually returns through an architecture-defined privileged return instruction. From the application's perspective, this often looks like a normal C function call because the C library hides the ABI details.

System calls are deliberately narrow. A process cannot call the kernel's internal VFS helper or scheduler function by address. It can invoke only exported operations, with arguments that cross a checked interface. This reduces the attack surface and allows kernel implementation details to change without recompiling every application.

#### **Exceptions and Faults**

An **exception** is tied to the instruction currently being executed. The instruction may be illegal, its operands may be invalid, or it may require operating-system intervention. Common examples include an invalid opcode, a protection violation, and a page-table entry that is absent.

The word **fault** usually describes a synchronous exception that may be handled so the instruction can be retried. Demand paging is the classic example: the virtual address is valid, but its page is not resident. The kernel obtains or creates the page, updates the mapping, and returns to the faulting instruction. The program experiences a longer memory access, not an explicit system call.

Terminology such as *fault*, *trap*, and *abort* differs across architectures and textbooks, so the useful questions are more concrete:

1. Which instruction and address caused the event?
2. Is the state precise enough to retry or resume?
3. Does the current process have a valid mapping or registered handler?
4. Should the kernel repair the condition, report it to user space, terminate the process, or stop because kernel integrity is compromised?

A user-space segmentation fault and a kernel page fault are therefore not equivalent. The first usually means one process violated its address-space rules. The second may indicate a bug in privileged code and can threaten the whole system, unless it occurred during a carefully guarded copy from user memory.

#### **Hardware Interrupts**

A **hardware interrupt** reports an event external to the current instruction stream. A timer expires, a network interface receives packets, or a storage controller completes a request. The interrupt controller routes the event to a CPU, and the CPU transfers control at an instruction boundary when the event is eligible for delivery.

Interrupt handlers must usually do little work on the immediate path. While an interrupt is being serviced, latency-sensitive work may be delayed and some interrupts may be masked. A common design is:

1. acknowledge the hardware so it can continue;
2. capture minimal completion state;
3. queue or mark deferred work;
4. wake threads waiting for the event;
5. return, allowing normal scheduling to decide when longer processing runs.

Interrupts make blocking I/O efficient. Instead of repeatedly asking whether a disk request has finished, a process can sleep. The CPU runs other work, and the device interrupt later lets the kernel mark the request complete and wake the process.

#### **Trap Entry and Return**

Trap handling is a cooperation between hardware, assembly-language entry code, and higher-level kernel code. The exact registers differ, but the conceptual path is stable:

1. **Finish or identify the interrupted instruction.** The architecture records a resume address and a cause code.
2. **Raise privilege and redirect control.** The CPU selects a previously configured trap vector. User code cannot provide the target at the moment of entry.
3. **Preserve user state.** Entry assembly saves general-purpose registers that the hardware did not save automatically.
4. **Establish kernel context.** The path selects a protected kernel stack and any required kernel address-space state.
5. **Build a trap frame.** Saved registers, status, cause, and resume information become a structured snapshot.
6. **Classify and handle the event.** Kernel code dispatches to a system-call routine, exception logic, or device driver.
7. **Prepare the result.** The kernel may update return registers, process state, mappings, pending signals, or scheduling state.
8. **Restore and return.** Entry state is restored, privilege is lowered, and execution resumes or starts at a kernel-selected user address.

The **trap frame** is crucial because entering the kernel does not erase the interrupted computation. It gives the kernel a representation of the user context that can be inspected, modified when semantics require it, and restored later. If the thread blocks, that restoration may occur long after another thread has run on the same CPU.

In xv6 on RISC-V, the user stub places the system-call number in `a7` and executes `ecall`. Trap entry uses the trampoline and saves registers into a per-process trap frame; `usertrap()` inspects the cause, and the kernel eventually prepares a return to user mode. This small path is an excellent concrete model, while Linux adds architecture support, tracing, security hooks, signals, preemption, and many more subsystems around the same core idea.

### **The System Call Interface**

The system-call interface is the most important explicit contract between ordinary programs and the kernel. It should be stable enough for binaries to keep working, small enough to audit, expressive enough to support libraries and runtimes, and defensive enough to process untrusted inputs.

![Linux system-call interface and the GNU C Library](assets/linux-system-call-interface.svg){fig-alt="Applications use the GNU C Library and other APIs, which invoke the Linux kernel system-call interface above hardware." width="72%"}

*Figure source: ScotXW, [Linux kernel System Call Interface and glibc](https://commons.wikimedia.org/wiki/File:Linux_kernel_System_Call_Interface_and_glibc.svg), CC BY-SA 3.0.*

#### **API, ABI, and Calling Convention**

An **API** is a source-level programming contract: function names, argument types, return meanings, and documented behavior. An **ABI** is a binary-level contract: register assignments, system-call numbers, data layouts, instruction choice, error encoding, and other details required for compiled code to interoperate.

For a call such as `write(fd, buffer, count)`, several layers are involved:

| Layer | Example responsibility | Can it change without changing the others? |
|---|---|---|
| Application API | Calls the POSIX-style `write()` function | Source can be recompiled against another compatible library |
| C library wrapper | Adapts C calling convention to kernel ABI; sets `errno` | Library implementation can change while preserving its API |
| System-call ABI | Places number and arguments in defined registers | Must remain stable for existing binaries |
| Kernel implementation | Validates descriptor and memory, then reaches VFS/driver logic | Internal organization can change behind the ABI |

On common 64-bit Linux targets, the essential register conventions are:

| Architecture | System-call instruction | Number | Arguments | Result |
|---|---|---|---|---|
| x86-64 | `syscall` | `rax` | `rdi`, `rsi`, `rdx`, `r10`, `r8`, `r9` | `rax` |
| RISC-V 64 | `ecall` | `a7` | `a0` through `a5` | `a0` |

The x86-64 system-call convention is not identical to the ordinary System V C function convention: the fourth system-call argument uses `r10`, while ordinary C calls use `rcx`. Library wrappers handle details like this. The Linux [`syscall(2)` manual](https://man7.org/linux/man-pages/man2/syscall.2.html) documents architecture-specific conventions, and [`syscalls(2)`](https://man7.org/linux/man-pages/man2/syscalls.2.html) distinguishes kernel calls from library wrapper functions.

The following Linux C program performs one write through the normal libc wrapper and another through libc's generic `syscall()` helper. The helper still belongs to libc, but it asks for a specific kernel system-call number and exposes the lower-level interface more directly.

<details>
<summary>Show the annotated C example</summary>

```c
#define _GNU_SOURCE
#include <errno.h>
#include <stdio.h>
#include <string.h>
#include <sys/syscall.h>
#include <unistd.h>

int main(void) {
    const char wrapped[] = "libc write wrapper\n";
    const char raw[] = "SYS_write through syscall()\n";

    // Normal source-level API. libc selects the correct kernel ABI.
    if (write(STDOUT_FILENO, wrapped, sizeof wrapped - 1) == -1) {
        perror("write");
        return 1;
    }

    // Lower-level request: explicitly provide Linux's SYS_write number.
    long result = syscall(
        SYS_write, STDOUT_FILENO, raw, sizeof raw - 1
    );
    if (result == -1) {
        // syscall() converts the kernel error convention into errno.
        fprintf(stderr, "SYS_write: %s\n", strerror(errno));
        return 1;
    }

    return 0;
}
```

```bash
cc -std=c11 -Wall -Wextra syscall_write.c -o syscall_write
strace -e trace=write ./syscall_write
```

</details>

Both operations should appear as `write` events in `strace`. The source-level paths differ, but they converge on the same kernel service. This also demonstrates why 'library call' and 'system call' are not synonyms: `strlen()` needs no kernel service, while the `write()` wrapper normally does.

#### **Validating Arguments Across the Trust Boundary**

At system-call entry, the kernel receives numbers and addresses supplied by untrusted code. A pointer is not an object that the kernel may blindly dereference; it is a numeric virtual address in the caller's address space. The mapping can be absent, read-only, too short, or changed concurrently by another thread.

![Validation path across the system-call trust boundary](assets/syscall-validation-path.svg){fig-alt="A system call moves from an application API through libc and the ABI into kernel dispatch, validation, a subsystem, and a protected return path." width="98%"}

*Figure: original diagram for this chapter, informed by Linux and xv6 system-call interfaces.*

For a request resembling `write(fd, buffer, count)`, kernel-side reasoning includes at least:

1. **Dispatch validation:** is the system-call number implemented for this architecture and process state?
2. **Scalar validation:** is `count` representable, bounded, and safe from arithmetic overflow?
3. **Handle validation:** does `fd` name an open object in this process, and is it writable?
4. **Pointer validation:** does the user range identify readable user pages for the requested length?
5. **Permission validation:** do credentials and object policy permit the operation?
6. **State validation:** is the object still open and in a state that accepts the operation?
7. **Safe transfer:** use guarded copy routines or pinned/mapped buffers rather than an unchecked kernel dereference.
8. **Result discipline:** report partial progress, interruption, or a precise error without leaking kernel data.

The check and the use must also be considered together. If the kernel checks a user-space pathname and later reads it again, another thread might modify the memory between those actions. Kernels reduce such **time-of-check to time-of-use** risks by copying data into kernel-owned memory, taking references or locks on resolved objects, or using interfaces whose semantics explicitly tolerate concurrent change.

User programs must respect the same contract from the other side. For byte-stream I/O, a successful `write()` may transfer fewer bytes than requested, and an interrupted operation may need to be retried. A robust loop tracks progress instead of assuming one call completes everything.

<details>
<summary>Show a robust user-space write loop</summary>

```c
#include <errno.h>
#include <stddef.h>
#include <unistd.h>

int write_all(int fd, const void *buffer, size_t length) {
    const unsigned char *bytes = buffer;
    size_t offset = 0;

    while (offset < length) {
        // Ask the kernel to consume only the bytes still pending.
        ssize_t written = write(fd, bytes + offset, length - offset);

        if (written > 0) {
            offset += (size_t)written;  // Preserve partial progress.
            continue;
        }

        if (written == -1 && errno == EINTR) {
            continue;                  // A signal interrupted the call.
        }

        return -1;                     // Keep errno from the real failure.
    }

    return 0;
}
```

</details>

The kernel's validation protects the system from the caller. The caller's return-value checks protect the application from incorrect assumptions about asynchronous and resource-constrained execution.

#### **System Call Cost and Security**

A protected transition costs more than an ordinary function call, but there is no single universal 'system-call cost.' It may include entry and return instructions, register saving, dispatch, security hooks, argument copying, cache effects, speculation mitigations, and the requested work itself. A system call that hits cached metadata can be far cheaper than one that blocks for storage or network I/O.

Common techniques reduce crossings or amortize their fixed cost:

| Technique | Idea | Example |
|---|---|---|
| User-space buffering | Combine many small operations | `stdio` collects characters before `write()` |
| Batching | Describe multiple operations per entry | vector I/O, batched message operations |
| Shared mappings | Exchange data through mapped pages | `mmap`, shared-memory queues |
| Asynchronous submission | Submit work and collect completions separately | Linux `io_uring` |
| Kernel-provided user mapping | Read selected safe data without a trap | Linux vDSO for some time-related functions |

Optimization must preserve semantics. Replacing many durable writes with one buffered write can improve throughput but may change what survives a crash. Sharing memory avoids copies but introduces synchronization and validation problems. The right question is not 'how do I eliminate every syscall?' but 'which boundary crossings dominate this workload, and what semantic trade-off is acceptable?'

The system-call surface is also a security surface. A smaller permitted set can reduce what a compromised process can ask the kernel to do. Linux mechanisms such as seccomp can restrict system calls, while credentials, capabilities, namespaces, and mandatory access controls constrain authority more broadly. Those mechanisms do not make kernel handlers safe by themselves: every allowed handler must still validate lengths, object lifetimes, pointer ranges, integer arithmetic, and concurrent state.

### **Kernel Organization**

All kernels must implement protected services, but they disagree about which services should execute in the most privileged address space. This choice changes communication cost, failure containment, trusted-computing-base size, and extensibility.

![Monolithic, microkernel, and hybrid kernel organization](assets/kernel-organization.svg){fig-alt="A side-by-side comparison of monolithic, microkernel, and hybrid kernel structures, showing which services run in user or kernel space." width="100%"}

*Figure source: Golftheman, [OS-structure2](https://commons.wikimedia.org/wiki/File:OS-structure2.svg), public domain.*

The labels in this diagram describe architectural tendencies, not rigid boxes. Production systems often borrow ideas from several designs, and the placement of one service does not determine every other design choice.

#### **Monolithic and Modular Kernels**

In a **monolithic kernel**, major services such as virtual memory, file systems, networking, scheduling, and many drivers execute in one privileged kernel address space. They can call one another through ordinary in-kernel function calls and share data structures efficiently.

Linux is commonly described as monolithic but modular. Loadable kernel modules let drivers or subsystems be added after boot, yet a loaded module normally has kernel privilege and shares the kernel's failure domain. Modularity improves organization and deployment; it does not turn a driver into an isolated user process.

Advantages include direct communication, mature shared infrastructure, and high performance on common paths. The cost is a large trusted computing base: a memory-safety or synchronization bug in a privileged driver can corrupt unrelated kernel state. Internal interfaces can also become tightly coupled because direct access is convenient.

xv6 is deliberately monolithic and small. That makes complete control paths readable: a system call can proceed from trap handling into the file system or process subsystem without crossing multiple protection domains. It is a teaching advantage, not a claim that a tiny design contains every production requirement.

#### **Layered Systems**

A **layered design** organizes services so that each layer uses lower-level interfaces and serves higher-level ones. A conceptual stack might place hardware support below interrupt and memory mechanisms, then process and I/O services, then file and network abstractions, and finally the system-call interface.

Layering helps humans reason about dependencies. A narrow interface makes it easier to replace or test an implementation, and lower layers do not need to understand every higher-level policy. Strict layering can nevertheless force unnatural paths, duplicate work, or prevent an optimization that needs information from another layer. Most general-purpose kernels therefore use layering as an organizing principle while permitting carefully controlled cross-layer interactions.

#### **Microkernels and Message Passing**

A **microkernel** keeps a smaller set of mechanisms in privileged mode, often including low-level address-space management, scheduling, interrupt handling, and interprocess communication. Services such as drivers, file systems, or network stacks can run as isolated user-space servers. A client requests a service by sending a message rather than calling a large in-kernel subsystem directly.

This organization can improve fault isolation and reduce the most privileged trusted computing base. A failed user-space driver may be restartable without corrupting the microkernel. Explicit messages also make authority and component boundaries easier to analyze.

The trade-off is that a logical operation may require extra domain crossings, scheduling decisions, and data transfers. High-performance microkernels reduce these costs with optimized IPC, shared-memory transfer, capability-based naming, careful scheduling, and designs that avoid unnecessary messages. Systems such as seL4 emphasize a small, analyzable kernel; QNX and MINIX illustrate service-oriented organization in different settings.

Message passing is not automatically slower in every workload, and monolithic code is not automatically faster once failures, maintainability, and security are considered. Architecture determines where costs and guarantees are paid; implementation quality and workload determine their practical weight.

#### **Hybrid, Exokernel, and Library-OS Ideas**

The term **hybrid kernel** is used for systems that retain many services in kernel space while adopting ideas associated with microkernels or layered component models. Windows NT and Apple's XNU are often described this way, although the label hides substantial internal differences. 'Hybrid' should therefore prompt examination of actual protection domains and communication paths rather than serve as a complete explanation.

An **exokernel** takes a different approach: keep protection and secure resource multiplexing in the kernel, but expose lower-level resources so application-specific library operating systems can build their own abstractions. A **library OS** packages operating-system services with an application or runtime. Unikernel-style systems push this idea toward a single-purpose image deployed on a virtual machine or similarly controlled platform.

These approaches can specialize aggressively and reduce general-purpose overhead, but they shift compatibility, debugging, update, sharing, and isolation responsibilities. They are useful reminders that familiar abstractions such as a POSIX process or file system are design choices, not physical laws.

| Organization | Privileged code | Communication style | Main strength | Main pressure point |
|---|---|---|---|---|
| Monolithic | Most core services and drivers | Direct in-kernel calls | Efficient, integrated common paths | Large shared failure and trust domain |
| Modular monolithic | Monolithic core plus loadable components | Direct calls after loading | Deployment flexibility with native speed | Modules still have kernel authority |
| Layered | Depends on implementation | Calls through ordered interfaces | Clear dependencies and reasoning | Rigid boundaries can add indirection |
| Microkernel | Minimal mechanisms | IPC among isolated servers | Small privileged base and fault isolation | IPC, scheduling, and coordination overhead |
| Hybrid | Many services remain privileged | Calls plus component/message mechanisms | Practical compatibility and mixed design | Boundaries and guarantees can be hard to summarize |
| Exokernel/library OS | Protection and multiplexing core | Application-selected libraries | Specialization and low-level control | Portability, sharing, and ecosystem complexity |

### **Booting an Operating System**

The kernel cannot initialize itself before any code is running. Booting is a staged transfer of control in which each environment establishes enough machine state to load and trust the next one.

#### **Firmware and the Bootloader**

After reset, a processor begins at an architecture-defined location with limited initialized state. Platform firmware configures enough hardware to locate a boot target. On a modern PC, UEFI firmware maintains boot options in NVRAM and its boot manager can load a UEFI application, commonly an operating-system loader, according to platform policy.

![Conceptual boot path from CPU reset to the first user process](assets/boot-to-user-space.svg){fig-alt="A left-to-right sequence from CPU reset through firmware, boot manager, early kernel initialization, core subsystems, and the first user process." width="98%"}

*Figure: original diagram for this chapter, based on the [UEFI 2.11 Boot Manager specification](https://uefi.org/specs/UEFI/2.11/03_Boot_Manager.html) and the Linux/xv6 initialization model.*

A bootloader may choose among kernels, load a kernel image and initial RAM file system into memory, supply a command line and hardware information, and transfer execution to the kernel's entry point. Some platforms let the firmware load a kernel image more directly, so 'firmware -> bootloader -> kernel' is a useful model rather than an invariant sequence.

**Secure Boot** adds verification to the control chain. Each trusted stage verifies an acceptable signature before transferring control. It helps prevent unauthorized boot code, but it is not the same as full-disk encryption: signature verification answers *is this code trusted?*, whereas encryption answers *can someone read the stored data without a key?*

Older BIOS-style boot and modern UEFI boot differ significantly in interfaces and disk conventions, yet the conceptual purpose is the same: move from a tiny initial environment to code capable of loading and initializing the operating system.

#### **Kernel Initialization and the First User Process**

Once control reaches the kernel, it cannot immediately run normal applications. It must construct the abstractions that those applications assume. A simplified Linux-style sequence is:

1. **Establish early execution state:** set up an initial stack, architecture state, and enough address translation to execute reliably.
2. **Discover CPUs and memory:** identify usable physical ranges, reserve kernel regions, and initialize allocators.
3. **Install exception and interrupt handling:** configure vectors, controllers, and timers so protected events can be handled.
4. **Initialize core subsystems:** create scheduler, virtual-memory, device, VFS, networking, and security state in dependency order.
5. **Find a root file system:** use built-in drivers or an initial RAM file system to locate and mount the persistent root.
6. **Create the first user context:** start PID 1 on Linux or `init` in xv6.
7. **Build ordinary user space:** PID 1 starts services, reaps orphaned children, and eventually supports terminals, logins, or a graphical session.

An **initramfs** solves an apparent dependency cycle. The real root file system may live on encrypted storage, a RAID set, a network device, or hardware whose modules are not yet available. A small file system loaded with the kernel provides temporary user-space tools and drivers that prepare the real root, then switch to it.

Linux distributions often use `systemd` as PID 1, but PID 1 is a role rather than a required program name. Other init systems exist, containers can have their own PID-namespace init process, and xv6 starts a much smaller `init` program. The invariant is that the kernel creates an initial user execution context from which the rest of user space can be constructed.

The running system exposes evidence about its own boot path. These commands are intended for Linux or WSL; availability and permissions differ by distribution.

<details>
<summary>Show commands for inspecting a Linux boot</summary>

```bash
# Arguments supplied to the running kernel.
cat /proc/cmdline

# Kernel messages from the current boot. Some systems restrict dmesg.
dmesg --human | less

# On systemd systems, show time attributed to firmware, loader, kernel,
# and user-space initialization, then inspect the critical dependency chain.
systemd-analyze
systemd-analyze critical-chain

# Query logs only from the current boot.
journalctl -b
```

</details>

These observations should not be confused with universal stages. `systemd-analyze` reports a systemd-based boot model; an embedded system, xv6, or another init system exposes different evidence.

### **Observing the User-Kernel Boundary**

The boundary is not only theoretical. Linux tracing and pseudo-filesystems let us observe requests and kernel-maintained process state without modifying the application.

#### **Tracing System Calls**

`strace` records system calls and signals for a process. It can reveal which files a program opens, where it blocks, how child processes are created, and which error code caused a failure. It reports the interface, not every internal kernel function.

For the earlier C program:

```bash
strace -e trace=write ./syscall_write
```

A typical trace contains two `write(1, ..., length)` calls followed by process exit. File descriptor `1` is standard output. The exact addresses, quoting, and auxiliary calls depend on architecture, libc, loader, and `strace` version.

For a compact quantitative view:

```bash
strace -c ./syscall_write
```

The summary counts calls and reports observed time. Tracing itself adds overhead and can change timing, especially for highly concurrent programs, so it is diagnostic evidence rather than an untouched performance measurement.

#### **Inspecting Kernel-Exposed Process State**

Linux mounts `/proc` as a pseudo-filesystem. Files under `/proc/<pid>` are generated from kernel state when read; they are not ordinary records stored on disk. Useful views include:

- `status`: identifiers, credentials, state, capabilities, memory summaries, and signal masks;
- `maps`: current virtual-memory regions and permissions;
- `fd/`: symbolic links representing the process's open file descriptors;
- `syscall`: the current system call and register-level arguments when access is permitted.

The following sequence creates a process that is likely blocked in a sleep-related system call, inspects it, then waits for clean completion.

<details>
<summary>Show process-state inspection commands</summary>

```bash
sleep 30 &
pid=$!

# Identity, state, threads, and a few memory/accounting fields.
grep -E '^(Name|State|Pid|PPid|Threads|VmRSS|voluntary_ctxt_switches)' \
    /proc/$pid/status

# Current system-call number and arguments, if kernel policy permits it.
cat /proc/$pid/syscall

# Descriptor numbers mapped to kernel-managed objects.
ls -l /proc/$pid/fd

# First few virtual-memory mappings and their permissions.
head /proc/$pid/maps

wait "$pid"
```

</details>

The [`proc_pid_status(5)` manual](https://man7.org/linux/man-pages/man5/proc_pid_status.5.html) documents these fields. Access can be restricted by credentials, ptrace policy, namespaces, or how `/proc` is mounted. That restriction is itself an example of the kernel enforcing isolation around diagnostic information.

### **Following the Running Case into the Kernel**

The shell pipeline combines nearly every boundary concept introduced in this chapter. The diagram shows causal relationships, not one fixed interleaving: the scheduler may run `cat`, `grep`, and the shell in many valid orders.

![End-to-end kernel path for the recurring shell pipeline](assets/pipeline-kernel-path.svg){fig-alt="The shell creates cat and grep processes, connects them through a kernel pipe, redirects output to result.txt, and waits for completion." width="100%"}

*Figure: original diagram for this chapter.*

The execution unfolds approximately as follows:

1. **The shell parses syntax in user space.** It recognizes a two-command pipeline and an output redirection. Parsing does not require the kernel unless the shell needs additional input or resources.
2. **The shell creates kernel objects.** `pipe2()` creates a pipe with read and write descriptors. `openat()` creates or truncates `result.txt` and returns another descriptor.
3. **The shell creates child execution contexts.** `fork()` or a related Linux primitive produces children whose descriptor tables initially refer to the same open kernel objects.
4. **Each child installs its standard streams.** `dup2()` makes `cat` standard output refer to the pipe's write end. It makes `grep` standard input refer to the pipe's read end and standard output refer to `result.txt`.
5. **Unused descriptor copies are closed.** This is semantic, not cosmetic. If any process retains an extra pipe write end, the reader may never observe end-of-file.
6. **`execve()` replaces program images.** One child becomes `cat`; the other becomes `grep`. The processes keep descriptors that were not marked close-on-exec.
7. **Data flows through kernel objects.** `cat` opens `input.txt`, repeatedly reads bytes, and writes them to the pipe. `grep` reads from the pipe, searches in user space, and writes matching data to the output file. Reads and writes may block, wake, or complete partially.
8. **Closure communicates completion.** When every write end is closed, `grep` eventually reads end-of-file. Both programs exit, and the shell waits for their termination status before accepting the next foreground command.

The pipe is not a direct memory pointer from `cat` to `grep`. It is a kernel-managed object reached through per-process descriptors. That indirection lets the kernel enforce access, coordinate blocking, preserve object lifetime while references exist, and wake the reader when data becomes available.

We can ask Linux to record the important boundary crossings:

<details>
<summary>Show a focused trace of the pipeline</summary>

```bash
printf 'kernel mode\nuser mode\nkernel boundary\n' > input.txt

strace -f -o pipeline.trace \
  -e trace=clone,fork,vfork,execve,wait4,pipe,pipe2,dup2,openat,read,write,close \
  sh -c 'cat input.txt | grep kernel > result.txt'

cat result.txt
less pipeline.trace
```

</details>

In the trace, look for relationships rather than expecting identical descriptor numbers:

- a pipe is created before the two programs exchange data;
- the output file is opened by the shell-side setup path;
- child processes duplicate descriptors onto `0` or `1`;
- `execve()` changes each child into the requested program;
- `cat` opens and reads `input.txt`, then writes to the pipe;
- `grep` reads from the pipe and writes matching lines to `result.txt`;
- the shell waits while the foreground pipeline runs.

Loader behavior, locale files, dynamic libraries, shell implementation, and scheduling can add calls and change their order. This variability is a lesson in itself: a system-call trace is one execution of a concurrent system, not the only legal execution.

### **Comparison and Summary**

The chapter's ideas form one chain:

```text
physical resources
    -> protected hardware mechanisms
    -> privileged kernel
    -> validated system-call interface
    -> process, memory, file, pipe, and device abstractions
    -> ordinary applications
```

| Concept | Question it answers | Key idea to retain |
|---|---|---|
| Abstraction | What stable object does software use? | Hide hardware detail behind a meaningful contract |
| Multiplexing | How do many activities share finite resources? | Map logical objects onto time or space slices |
| Isolation | Why can one failure remain local? | Hardware protection plus kernel validation |
| Mechanism | What operation can the system enforce? | Trap, map, schedule, copy, block, wake |
| Policy | Which allowed choice should be made? | Select according to goals such as fairness, latency, or security |
| System call | How does a program intentionally request privilege? | Enter through a numbered, validated ABI |
| Exception | How does the current instruction request or require intervention? | Diagnose, repair and retry, or report failure |
| Interrupt | How does external hardware get attention? | Asynchronous entry followed by minimal handling and deferred work |
| Kernel architecture | Where should trusted services execute? | Trade communication cost against isolation and trusted-base size |
| Boot | How does an inert machine reach user space? | Staged transfer of control and, optionally, trust |

Several misconceptions are worth removing early:

| Misconception | More accurate statement |
|---|---|
| 'The operating system and kernel are identical.' | The kernel is the privileged core; the broader OS environment includes user-space libraries, tools, and services. |
| 'Every library function is a system call.' | Many functions run entirely in user space; wrappers enter the kernel only when a protected service is needed. |
| 'A system call always switches processes.' | It switches privilege; a process switch occurs only if scheduling or blocking requires one. |
| 'All kernel entries are system calls.' | Exceptions and hardware interrupts also enter privileged handlers. |
| 'A loadable driver is isolated because it is modular.' | A normal kernel module still shares kernel privilege and failure scope. |
| 'Fewer system calls always means faster code.' | Work, blocking, copying, caching, batching, and required semantics dominate real performance. |
| 'Boot ends when the kernel starts.' | The kernel must initialize subsystems, mount a root, and create the first user process before ordinary services exist. |

The most useful invariant is this: **user code never receives authority merely by asking for it**. Hardware provides controlled entry and protected state; the kernel interprets a request, validates it against the caller and current objects, applies policy, performs work, and returns a constrained result. Every later chapter builds a richer abstraction on this same boundary.

[Next: Processes and Program Execution](02-processes-and-program-execution.html)
